# Assignment 4: Neural Machine Translation with Transformers

## Learning Objectives
Implement a sequence-to-sequence Transformer model for French → English neural machine translation using PyTorch.

**Key components implemented here:**
- `MultiHeadAttention` — scaled dot-product attention with causal and padding masks
- `TransformerEmbeddings` — token + learned positional embeddings with weight tying
- `EncoderDecoderModel` — full encoder-decoder Transformer
- `map_example` — tokenization preprocessing
- `plot_attention_matrix` — attention heatmap visualization

> **Note:** Data and tokenizers are provided with the assignment. Run the setup cell first.


In [ ]:
# !pip install transformers==4.27.0 datasets==2.10.0 -q
# !wget --quiet https://princeton-nlp.github.io/cos484/assignments/a4/resources.zip
# !unzip -qo resources.zip

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
from typing import Optional, Dict, List, Tuple
from datasets import load_from_disk
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Data Preprocessing

In [ ]:
raw_text_datasets = load_from_disk("resources/parallel_en_fr_corpus")
print("Dataset splits:", raw_text_datasets)
print("First training example:", raw_text_datasets["train"][0])

In [ ]:
source_tokenizer = AutoTokenizer.from_pretrained("resources/tokenizer_fr")
target_tokenizer = AutoTokenizer.from_pretrained("resources/tokenizer_en")

print("Source vocab size:", source_tokenizer.vocab_size)
print("Target vocab size:", target_tokenizer.vocab_size)

# Tokenisation demo
example_sentence = "we have an example"
out = target_tokenizer(example_sentence)
decoded = [target_tokenizer.decode(tok) for tok in out["input_ids"]]
print("\nTokens:", decoded)
print("Reconstructed:", "".join(decoded).replace("▁", " "))

In [ ]:
def map_example(example: Dict[str, str]) -> Dict[str, List[int]]:
    """Tokenise a single parallel (fr, en) example.

    The Princeton COS484 dataset stores sentences in 'fr' and 'en' columns.
    source_tokenizer handles French; target_tokenizer handles English.
    Both tokenizers add BOS/EOS automatically.
    """
    # Detect column names — handle both direct ('fr'/'en') and nested ('translation') layouts
    if "fr" in example and "en" in example:
        src_text, tgt_text = example["fr"], example["en"]
    elif "translation" in example:
        src_text = example["translation"]["fr"]
        tgt_text = example["translation"]["en"]
    else:
        # Fallback: print the keys so the user can adjust
        raise KeyError(f"Unexpected dataset columns: {list(example.keys())}. "
                       "Edit map_example to use the correct column names.")

    encoder_input_ids = source_tokenizer(src_text)["input_ids"]
    decoder_input_ids = target_tokenizer(tgt_text)["input_ids"]
    return {"encoder_input_ids": encoder_input_ids, "decoder_input_ids": decoder_input_ids}

tokenized_datasets = raw_text_datasets.map(map_example, batched=False)
tokenized_datasets = tokenized_datasets.remove_columns(raw_text_datasets.column_names["train"])

assert set(tokenized_datasets.column_names["train"]) == {"decoder_input_ids", "encoder_input_ids"}
assert len(tokenized_datasets["train"]) == len(raw_text_datasets["train"])
print("Tokenisation complete — columns:", tokenized_datasets.column_names["train"])

## 2. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,
                 hidden_size: int,
                 num_attention_heads: int,
                 is_causal_attention: bool = False,
                 is_cross_attention: bool = False):
        """
        Flexible multi-head attention supporting:
        - Full self-attention (encoder)
        - Causal masked self-attention (decoder)
        - Cross-attention (decoder attending to encoder outputs)
        """
        super().__init__()
        assert hidden_size % num_attention_heads == 0,             "hidden_size must be divisible by num_attention_heads"
        self.num_heads          = num_attention_heads
        self.head_dim           = hidden_size // num_attention_heads
        self.hidden_size        = hidden_size
        self.is_causal          = is_causal_attention
        self.is_cross           = is_cross_attention

        # Separate projections for Q, K, V and the output
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.o_proj = nn.Linear(hidden_size, hidden_size)

    def causal_attention_mask(self,
                              sequence_length: int,
                              device: Optional[torch.device] = None) -> torch.FloatTensor:
        """Upper-triangular mask that prevents attending to future positions.

        Returns shape (1, 1, T, T) so it broadcasts over batch and head dims.
        We use -1e6 instead of -inf to avoid NaN in softmax when a row is all -inf.
        """
        # triu with diagonal=1 zeros out the diagonal and below, keeping only the upper triangle
        mask = torch.triu(torch.full((sequence_length, sequence_length), -1e6), diagonal=1)
        return mask.unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, T, T)

    def forward(self,
                hidden_states: torch.FloatTensor,
                key_padding_mask: torch.BoolTensor,
                encoder_outputs: Optional[torch.FloatTensor] = None,
                ) -> Tuple[torch.FloatTensor, torch.FloatTensor]:
        """
        Args:
            hidden_states:    (B, T_q, H) — decoder inputs (or encoder inputs for self-attn)
            key_padding_mask: (B, T_k)    — True where tokens are padding (should be ignored)
            encoder_outputs:  (B, T_enc, H) — only for cross-attention

        Returns:
            layer_output:     (B, T_q, H)
            attention_weights:(B, num_heads, T_q, T_k)
        """
        B, T_q, H = hidden_states.shape

        # For cross-attention K/V come from the encoder; otherwise self-attention
        kv_src = encoder_outputs if self.is_cross else hidden_states
        T_k    = kv_src.shape[1]

        # --- Project Q, K, V ---
        Q = self.q_proj(hidden_states)  # (B, T_q, H)
        K = self.k_proj(kv_src)         # (B, T_k, H)
        V = self.v_proj(kv_src)         # (B, T_k, H)

        # --- Reshape for multi-head: (B, T, H) → (B, n_heads, T, head_dim) ---
        Q = Q.reshape(B, T_q, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, T_k, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, T_k, self.num_heads, self.head_dim).transpose(1, 2)

        # --- Scaled dot-product scores: (B, n_heads, T_q, T_k) ---
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # --- Causal masking (adds large negatives above the diagonal) ---
        if self.is_causal:
            scores = scores + self.causal_attention_mask(T_q, device=hidden_states.device)

        # --- Padding mask: (B, T_k) → (B, 1, 1, T_k) broadcasts to (B, n_heads, T_q, T_k) ---
        pad_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
        scores   = scores.masked_fill(pad_mask, -1e6)

        # --- Normalise ---
        attention_weights = torch.softmax(scores, dim=-1)   # (B, n_heads, T_q, T_k)

        # --- Aggregate values ---
        out = torch.matmul(attention_weights, V)             # (B, n_heads, T_q, head_dim)

        # --- Merge heads and project ---
        out = out.transpose(1, 2).reshape(B, T_q, H)
        return self.o_proj(out), attention_weights

### 2.1 Sanity Checks

In [ ]:
embed_dim = 8; n_heads = 2; B = 1; enc_len = 5; dec_len = 7

enc_out = torch.randn(B, enc_len, embed_dim)
dec_in  = torch.randn(B, dec_len, embed_dim)
enc_pad = torch.zeros(B, enc_len, dtype=torch.bool); enc_pad[:, -1] = True
dec_pad = torch.zeros(B, dec_len, dtype=torch.bool); dec_pad[:, -2:] = True

cross  = MultiHeadAttention(embed_dim, n_heads, is_cross_attention=True)
causal = MultiHeadAttention(embed_dim, n_heads, is_causal_attention=True)

cross_out,  cross_w  = cross (dec_in, enc_pad, enc_out)
causal_out, causal_w = causal(dec_in, dec_pad)

assert cross_out.shape  == (B, dec_len, embed_dim)
assert cross_w.shape    == (B, n_heads, dec_len, enc_len)
assert causal_out.shape == (B, dec_len, embed_dim)
assert causal_w.shape   == (B, n_heads, dec_len, dec_len)

assert torch.isclose(cross_w.sum(-1),  torch.tensor(1.0)).all(), "Cross-attention weights don't sum to 1"
assert torch.isclose(causal_w.sum(-1), torch.tensor(1.0)).all(), "Causal-attention weights don't sum to 1"

# Masked positions should receive zero attention weight
assert torch.isclose(cross_w[:, :, :, -1],  torch.tensor(0.0)).all(),  "Encoder padding not masked"
assert torch.isclose(causal_w[:, :, :, -2:], torch.tensor(0.0)).all(), "Decoder padding not masked"
assert torch.isclose(causal_w[:, :, 2, 3:],  torch.tensor(0.0)).all(), "Causal mask not applied"

print("All attention sanity checks passed!")

### 2.2 Attention Heatmaps

In [ ]:
def plot_attention_matrix(attention_matrix, title):
    """Plot a 2-D attention weight matrix as a heatmap.

    Args:
        attention_matrix: numpy array (n_query_tokens, n_key_tokens)
        title: figure title
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(attention_matrix, cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel("key token position")
    ax.set_ylabel("query token position")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_attention_matrix(cross_w[0, 0].detach().numpy(),  "cross-attention, head 1")
plot_attention_matrix(cross_w[0, 1].detach().numpy(),  "cross-attention, head 2")
plot_attention_matrix(causal_w[0, 0].detach().numpy(), "causal self-attention, head 1")
plot_attention_matrix(causal_w[0, 1].detach().numpy(), "causal self-attention, head 2")

## 3. Transformer Embeddings

In [ ]:
class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, max_sequence_length: int):
        """Token embeddings + learned positional embeddings.

        Weight tying: the same embedding table is reused in compute_logits
        to project decoder outputs back to vocabulary logits.
        """
        super().__init__()
        self.token_embeddings    = nn.Embedding(vocab_size, hidden_size)
        self.position_embeddings = nn.Embedding(max_sequence_length, hidden_size)
        self.hidden_size         = hidden_size

    def compute_logits(self, decoder_output: torch.FloatTensor) -> torch.FloatTensor:
        """Project decoder hidden states to vocabulary logits (weight-tied).

        Args:
            decoder_output: (B, T, hidden_size)
        Returns:
            logits:         (B, T, vocab_size)
        """
        # F.linear(x, W) computes x @ W.T, which is the transposed embedding lookup
        return F.linear(decoder_output, self.token_embeddings.weight)

    def forward(self, input_ids: torch.LongTensor) -> torch.FloatTensor:
        """Sum token and position embeddings.

        Args:
            input_ids: (B, T)
        Returns:
            embeddings: (B, T, hidden_size)
        """
        B, T = input_ids.shape
        # arange gives [0, 1, ..., T-1]; unsqueeze(0) broadcasts over batch
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)  # (1, T)
        return self.token_embeddings(input_ids) + self.position_embeddings(positions)

## 4. Transformer Block (provided)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, hidden_size, intermediate_size, num_attention_heads,
                 hidden_dropout_prob, is_decoder=False):
        super().__init__()
        self.is_decoder = is_decoder
        self.self_attention            = MultiHeadAttention(hidden_size, num_attention_heads,
                                                            is_causal_attention=is_decoder)
        self.self_attention_layer_norm = nn.LayerNorm(hidden_size)
        if is_decoder:
            self.cross_attention           = MultiHeadAttention(hidden_size, num_attention_heads,
                                                                is_cross_attention=True)
            self.cross_attention_layer_norm = nn.LayerNorm(hidden_size)
        self.feedforward = nn.Sequential(
            nn.Linear(hidden_size, intermediate_size), nn.ReLU(),
            nn.Linear(intermediate_size, hidden_size), nn.Dropout(hidden_dropout_prob))
        self.feedforward_layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states, padding_mask, encoder_outputs=None, encoder_padding_mask=None):
        hidden_states = (self.self_attention(hidden_states, padding_mask)[0]
                         + hidden_states)
        hidden_states = self.self_attention_layer_norm(hidden_states)
        if self.is_decoder:
            hidden_states = (self.cross_attention(hidden_states, encoder_padding_mask, encoder_outputs)[0]
                             + hidden_states)
            hidden_states = self.cross_attention_layer_norm(hidden_states)
        hidden_states = self.feedforward(hidden_states) + hidden_states
        return self.feedforward_layer_norm(hidden_states)

## 5. Encoder-Decoder Model

In [ ]:
class EncoderDecoderModel(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, hidden_size,
                 intermediate_size, num_attention_heads, num_encoder_layers,
                 num_decoder_layers, max_sequence_length, hidden_dropout_prob):
        super().__init__()

        # Separate embedding tables for source and target vocabularies
        self.source_embeddings = TransformerEmbeddings(source_vocab_size, hidden_size, max_sequence_length)
        self.target_embeddings = TransformerEmbeddings(target_vocab_size, hidden_size, max_sequence_length)

        block_kwargs = dict(hidden_size=hidden_size, intermediate_size=intermediate_size,
                            num_attention_heads=num_attention_heads,
                            hidden_dropout_prob=hidden_dropout_prob)
        self.encoder_layers = nn.ModuleList([
            TransformerBlock(**block_kwargs, is_decoder=False)
            for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            TransformerBlock(**block_kwargs, is_decoder=True)
            for _ in range(num_decoder_layers)
        ])

    def forward_encoder(self, input_ids: torch.LongTensor,
                        padding_mask: torch.BoolTensor) -> torch.FloatTensor:
        """Run the encoder stack.

        Args:
            input_ids:    (B, T_src) source token ids
            padding_mask: (B, T_src) True where tokens are padding
        Returns:
            encoder hidden states (B, T_src, hidden_size)
        """
        hidden_states = self.source_embeddings(input_ids)
        for layer in self.encoder_layers:
            hidden_states = layer(hidden_states, padding_mask)
        return hidden_states

    def forward_decoder(self, input_ids, padding_mask, encoder_outputs, encoder_padding_mask):
        """Run the decoder stack and project to vocabulary logits.

        Args:
            input_ids:            (B, T_tgt) target token ids (shifted right)
            padding_mask:         (B, T_tgt) True where tokens are padding
            encoder_outputs:      (B, T_src, hidden_size)
            encoder_padding_mask: (B, T_src) True where encoder tokens are padding
        Returns:
            logits (B, T_tgt, target_vocab_size)
        """
        hidden_states = self.target_embeddings(input_ids)
        for layer in self.decoder_layers:
            hidden_states = layer(hidden_states, padding_mask,
                                  encoder_outputs, encoder_padding_mask)
        # Weight-tied projection back to vocabulary
        return self.target_embeddings.compute_logits(hidden_states)

    def forward(self, encoder_input_ids, encoder_padding_mask,
                decoder_input_ids, decoder_padding_mask):
        enc_out = self.forward_encoder(encoder_input_ids, encoder_padding_mask)
        return self.forward_decoder(decoder_input_ids, decoder_padding_mask,
                                    enc_out, encoder_padding_mask)

## 6. Training

In [ ]:
def collate_fn(examples: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
    enc_len = max(len(e["encoder_input_ids"]) for e in examples)
    dec_len = max(len(e["decoder_input_ids"]) for e in examples)
    B = len(examples)

    enc_ids  = torch.full((B, enc_len), source_tokenizer.pad_token_id, dtype=torch.int64)
    enc_mask = torch.ones ((B, enc_len), dtype=torch.bool)
    dec_ids  = torch.full((B, dec_len), target_tokenizer.pad_token_id, dtype=torch.int64)
    dec_mask = torch.ones ((B, dec_len), dtype=torch.bool)

    for i, ex in enumerate(examples):
        el = len(ex["encoder_input_ids"]); dl = len(ex["decoder_input_ids"])
        enc_ids[i, :el]  = torch.tensor(ex["encoder_input_ids"])
        enc_mask[i, :el] = False
        dec_ids[i, :dl]  = torch.tensor(ex["decoder_input_ids"])
        dec_mask[i, :dl] = False

    return {"encoder_input_ids": enc_ids, "encoder_padding_mask": enc_mask,
            "decoder_input_ids": dec_ids, "decoder_padding_mask": dec_mask}

In [ ]:
def compute_loss_per_token(model, batch):
    logits = model(**batch)
    valid  = ~(batch["decoder_padding_mask"][:, 1:])
    labels = batch["decoder_input_ids"][:, 1:][valid]
    lgs    = logits[:, :-1][valid]
    return F.cross_entropy(lgs, labels, reduction='none')

def evaluate_perplexity(model, dataset, batch_size=32, device="cpu"):
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)
    loss_sum = n_tok = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            tl = compute_loss_per_token(model, batch)
            loss_sum += tl.sum(); n_tok += tl.numel()
    return (loss_sum / n_tok).exp().cpu().item()

def train(model, train_ds, val_ds, batch_size=32, lr=1e-3, max_epoch=15,
          log_every=10, valid_niter=100, model_path="model.pt"):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(dev); model.train()
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    step = epoch = loss_sum = n_tok = n_ex = 0
    best_ppl = float('inf')
    t_step = t0 = time.time()
    print(f"Training on {dev}")

    while True:
        loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size,
                                             shuffle=True, collate_fn=collate_fn)
        epoch += 1
        for i, batch in enumerate(loader):
            step += 1
            batch = {k: v.to(dev) for k, v in batch.items()}
            opt.zero_grad()
            tl = compute_loss_per_token(model, batch)
            (tl.sum() / batch_size).backward()
            opt.step()
            loss_sum += tl.sum().item(); n_tok += tl.numel(); n_ex += batch_size

            if step % log_every == 0:
                print(f"ep {epoch} step {step} | nll={loss_sum/n_ex:.2f} "
                      f"ppl={math.exp(loss_sum/n_tok):.2f} "
                      f"tok/s={n_tok/(time.time()-t_step):.0f}")
                loss_sum = n_tok = n_ex = 0; t_step = time.time()

            if step % valid_niter == 0:
                ppl = evaluate_perplexity(model, val_ds, batch_size=batch_size, device=dev)
                print(f"  [valid] step {step} ppl={ppl:.2f}")
                if ppl < best_ppl:
                    best_ppl = ppl; torch.save(model.state_dict(), model_path)
                    print(f"  Saved best model (ppl={ppl:.2f})")
                model.train()

        if epoch == max_epoch:
            print("Max epochs reached"); break

In [ ]:
torch.manual_seed(42); torch.cuda.manual_seed(42)

model = EncoderDecoderModel(
    source_vocab_size    = source_tokenizer.vocab_size,
    target_vocab_size    = target_tokenizer.vocab_size,
    hidden_size          = 32,
    intermediate_size    = 128,
    num_attention_heads  = 4,
    num_encoder_layers   = 3,
    num_decoder_layers   = 3,
    max_sequence_length  = 32,
    hidden_dropout_prob  = 0.1,
)
print("Architecture:", model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

train(model, tokenized_datasets["train"], tokenized_datasets["validation"], max_epoch=15)

## 7. Evaluation — Beam Search & BLEU

In [ ]:
def beam_search(model, encoder_input_ids, beam_width=5, max_len=32):
    model.eval()
    enc = encoder_input_ids.unsqueeze(0)
    enc_mask = torch.zeros_like(enc, dtype=torch.bool)
    enc_out  = model.forward_encoder(enc, enc_mask)

    generations = [torch.tensor([target_tokenizer.bos_token_id], device=enc.device)]
    scores      = [0.0]
    best_gen, best_score = None, float('-inf')

    for _ in range(max_len):
        new_gens, new_scores = [], []
        for score, gen in zip(scores, generations):
            gen_b   = gen.unsqueeze(0)
            pad_b   = torch.zeros_like(gen_b, dtype=torch.bool)
            dec_out = model.forward_decoder(gen_b, pad_b, enc_out, enc_mask)
            lp      = dec_out[0, -1].log_softmax(-1)
            top_v, top_i = lp.topk(beam_width)
            new_gens.extend([torch.cat([gen_b.squeeze(0), top_i[[k]]]) for k in range(beam_width)])
            new_scores.extend([(score + top_v[k].item()) for k in range(beam_width)])

        eos_id = target_tokenizer.eos_token_id
        for s, g in zip(new_scores, new_gens):
            if g[-1] == eos_id and s > best_score:
                best_score, best_gen = s, g

        if best_score >= max(new_scores): break
        top_idx    = sorted(range(len(new_scores)), key=lambda i: new_scores[i], reverse=True)[:beam_width]
        generations = [new_gens[i]  for i in top_idx]
        scores      = [new_scores[i] for i in top_idx]

    if best_gen is None:
        best_gen, best_score = generations[0], scores[0]
    return best_gen, best_score

def run_generation(model, test_dataset, beam=5, max_len=32):
    dev  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load("model.pt", map_location=dev))
    model = model.to(dev).eval()
    inputs, refs, cands = [], [], []
    with torch.no_grad():
        for ex in tqdm(test_dataset):
            enc_ids = torch.tensor(ex["encoder_input_ids"], device=dev)
            gen, _  = beam_search(model, enc_ids, beam, max_len)
            inputs.append("".join(source_tokenizer.decode(t).replace("▁"," ") for t in ex["encoder_input_ids"][1:-1]))
            refs.append  ("".join(target_tokenizer.decode(t).replace("▁"," ") for t in ex["decoder_input_ids"][1:-1]))
            cands.append ("".join(target_tokenizer.decode(t).replace("▁"," ") for t in gen[1:-1].cpu()))

    from nltk.translate.bleu_score import corpus_bleu
    bleu = corpus_bleu([[r] for r in refs], cands)
    return bleu, inputs, refs, cands

bleu, inputs, refs, cands = run_generation(model, tokenized_datasets["test"])
print(f"\nCorpus BLEU: {bleu*100:.2f}")

In [ ]:
# Show sample translations
for k in range(10, 20):
    print(f"===== Sample {k} =====")
    print(f"Input: {inputs[k]}")
    print(f"Gold:  {refs[k]}")
    print(f"Pred:  {cands[k]}")